In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from lora import inject_lora_layers
from datasets import load_dataset
from helper_functions import tokenize, format_prompt
from torch.utils.data import DataLoader
import wandb

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Loading the model
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto"
)

device = next(model.parameters()).device
print(f"Model is on device: {device}")

In [ ]:
# See all modules and their types
for name, module in model.named_modules():
    if name.find("q_proj") != -1 or name.find("v_proj") != -1:
        print(name, "→", type(module).__name__)

In [ ]:
# Freeze all model parameters first.
for param in model.parameters():
        param.requires_grad = False

# Then inject LoRA layers into the model
target_modules = ["q_proj", "v_proj"]
model = inject_lora_layers(model, rank=4, alpha=8, dropout=0.05, target_modules=target_modules)

In [ ]:
# Check if the LoRA layers are correctly injected
for name, module in model.named_modules():
    if name.find("q_proj") != -1 or name.find("v_proj") != -1:
        print(name, "→", type(module).__name__)

In [ ]:
# Count total and trainable parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")

In [ ]:
# Check if the LoRA layers are trainable while the others are frozen
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

In [ ]:
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Limit the dataset to 10k samples
dataset = dataset.shuffle(seed=42).select(range(10_000))

# Pull an example from the dataset to see its structure
print(dataset[0])

dataset = dataset.map(format_prompt)
tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")
train_dataloader = DataLoader(tokenized_dataset, batch_size=4, shuffle=True)

In [ ]:
# Initialize Weights & Biases for experiment tracking
# Make sure to replace "project" and "entity" with your actual W&B entity and project names.
wandb.init(
    project="medical-qlora",
    entity="dogukansagir-none",
    config={
        "model": "mistralai/Mistral-7B-Instruct-v0.3",
        "dataset": "ChatDoctor-HealthCareMagic-100k",
        "samples": 10_000,
        "rank": 4,
        "alpha": 8,
        "dropout": 0.05,
        "learning_rate": 1e-4,
        "epochs": 1,
        "batch_size": 4,
        "max_length": 512,
    }
)

In [ ]:
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
num_epochs = 1
total_steps = len(train_dataloader) * num_epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

model.gradient_checkpointing_enable()
model.train()
for epoch in range(num_epochs):
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            print(f"Epoch {epoch} | Step {step}/{total_steps} | Loss {loss.item():.4f}")
            wandb.log({
                "loss": loss.item(),
                "learning_rate": scheduler.get_last_lr()[0],
                "epoch": epoch,
                "step": step,
            })
        # Save checkpoint every 200 steps
        if step % 200 == 0 and step > 0:
            checkpoint_path = f"checkpoint_step_{step}.pt"
            trainable_state = {k: v for k, v in model.state_dict().items() if v.requires_grad}
            torch.save({
                "step": step,
                "model_state_dict": trainable_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": loss.item(),
            }, checkpoint_path)
            print(f"Checkpoint saved → {checkpoint_path}")

print("Training complete!")
wandb.finish()

# Save final LoRA adapter weights only
lora_state_dict = {
    k: v for k, v in model.state_dict().items() if "lora_" in k
}
torch.save(lora_state_dict, "lora_adapter.pt")
print("LoRA adapter saved → lora_adapter.pt")

In [ ]:
# Inference code (for later use)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from lora import inject_lora_layers
import gradio as gr


# Same 4‑bit Quantization Config as training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model (without LoRA)
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto"
)

# Freeze all parameters (same as training)
for param in model.parameters():
    param.requires_grad = False

# Inject LoRA layers with the same hyperparameters as training
target_modules = ["q_proj", "v_proj"]
model = inject_lora_layers(model, rank=4, alpha=8, dropout=0.05, target_modules=target_modules)

# Load the saved LoRA weights
lora_state_dict = torch.load("lora_adapter.pt")
model.load_state_dict(lora_state_dict, strict=False)  # strict=False because base layers are missing

# Move the whole model to the correct device (if needed)
device = next(model.parameters()).device
model = model.to(device)
model.eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def chat(message):
    device = next(model.parameters()).device
    prompt = f"<s>[INST] You are a helpful medical assistant. Never make a diagnosis. Only suggest possible explanations and always recommend consulting a real doctor. {message} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.2,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,      # stop at </s>
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode the whole sequence
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Find the last occurrence of [/INST] and take everything after it
    prompt_end = full_output.rfind("[/INST]") + len("[/INST]")
    answer = full_output[prompt_end:].strip()
    # Remove any trailing </s> if present
    answer = answer.replace("</s>", "").strip()
    return answer

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Your medical question"),
    outputs=gr.Textbox(label="Medical Assistant"),
    title="Medical Assistant — QLoRA fine-tuned Mistral 7B",
    description="Fine-tuned on ChatDoctor-HealthCareMagic-100k dataset"
).launch()